# 10 — Elyra：檢查輸入 + config（須滿足 `11` 推論）

核對 Input 是否推論可用。`INPUT_MODE=existing` **不會**自動還原 backup，若檔案仍是三日 smoke（~21K）請先：

```bash
cp -f .../CorrdiffInput_EC_RAW_{DATE}.nc.full-backup.nc \
      .../CorrdiffInput_EC_RAW_{DATE}.nc
```

| 環境變數 | 預設 | 說明 |
|----------|------|------|
| `DATE` / `SHOME` | `20260707` / `/mnt/corrdiff` | |
| `REQUIRE_INFERENCE_READY` | `1` | `0`=只檢查檔案存在 |
| `MIN_INPUT_BYTES` | `100000` | 無 netCDF4 時用檔案大小當閘門（smoke~21K，full~500K+） |


In [ ]:
import os
from pathlib import Path
import subprocess

SHOME = os.environ.get("SHOME", "/mnt/corrdiff")
DATE = os.environ.get("DATE", "20260707")
REQUIRE_INFERENCE_READY = os.environ.get("REQUIRE_INFERENCE_READY", "1").strip() != "0"
MIN_INPUT_BYTES = int(os.environ.get("MIN_INPUT_BYTES", "100000"))

print(f"SHOME={SHOME} DATE={DATE} REQUIRE_INFERENCE_READY={REQUIRE_INFERENCE_READY}")
workdir = Path(SHOME) / "workdir" / DATE
input_nc = workdir / f"CorrdiffInput_EC_RAW_{DATE}.nc"
backup_nc = Path(str(input_nc) + ".full-backup.nc")

for rel in ("bin", "config", "etc", f"workdir/{DATE}"):
    p = Path(SHOME) / rel
    print(f"[{'OK' if p.exists() else 'MISS'}] {p}")

assert input_nc.is_file(), (
    f"Missing input: {input_nc}. "
    "Restore *.full-backup.nc or re-seed / INPUT_MODE=url."
)
print(f"[OK] input {input_nc} ({input_nc.stat().st_size} bytes)")
if backup_nc.is_file():
    print(f"[OK] backup {backup_nc} ({backup_nc.stat().st_size} bytes)")


In [ ]:
workdir.mkdir(parents=True, exist_ok=True)
config_out = workdir / "config.yaml"
gen = Path(SHOME) / "bin" / "config_gen.py"
gen_cfg = Path(SHOME) / "config" / "gen_config.yaml"
assert gen.is_file(), f"Missing {gen}"
assert gen_cfg.is_file(), f"Missing {gen_cfg}"

subprocess.run(
    [
        "python3", str(gen),
        "-i", str(gen_cfg),
        "-o", str(config_out),
        "-v", f"dtg={DATE}",
        "-v", f"SHOME={SHOME}",
    ],
    check=True,
)
print(f"[OK] config.yaml -> {config_out}")

for name in (
    "UNet.Under850_v1.mdlus",
    "EDMPrecondSR.Under850_v1.mdlus",
    "ERA_TREAD_19910101_20231231_normalize_parameter.txt",
    "wrf_208x208_grid_coords.nc",
):
    p = Path(SHOME) / "etc" / name
    print(f"[{'OK' if p.is_file() else 'MISS'}] etc/{name}")
    assert p.is_file(), f"Missing model asset: {p}"

import yaml

cfg = yaml.safe_load(config_out.read_text())
fc = os.environ.get("FC_VERSION", "ec46day")
op = os.environ.get("OP_VERSION", "opv1")
need_lead = int(cfg[fc][op]["num_leadday"])
print(f"[OK] config {fc}/{op} num_leadday={need_lead}")

restore_hint = (
    f"cp -f {backup_nc} {input_nc}"
    if backup_nc.is_file()
    else "Place a full CorrdiffInput (lead≈45) then re-run."
)

if REQUIRE_INFERENCE_READY:
    have_lead = None
    try:
        import netCDF4 as nc

        with nc.Dataset(str(input_nc)) as ds:
            dims = {k: len(v) for k, v in ds.dimensions.items()}
        print("[OK] Input NC dims:", dims)
        have_lead = int(dims.get("lead", 0))
    except ImportError:
        print("[WARN] netCDF4 not in runtime — using file-size gate only")

    size = input_nc.stat().st_size
    if have_lead is not None and have_lead < need_lead:
        raise AssertionError(
            f"Input not inference-ready: lead={have_lead} < num_leadday={need_lead}. "
            f"INPUT_MODE=existing only reuses the current file (still smoke?). Restore: {restore_hint}"
        )
    if size < MIN_INPUT_BYTES:
        raise AssertionError(
            f"Input not inference-ready: size={size} < MIN_INPUT_BYTES={MIN_INPUT_BYTES} "
            f"(3-day smoke is ~21K). INPUT_MODE=existing does NOT restore backup. "
            f"Restore first: {restore_hint}"
        )
    if have_lead is not None:
        print(f"[OK] lead={have_lead} >= num_leadday={need_lead}")
    print(f"[OK] size={size} >= MIN_INPUT_BYTES={MIN_INPUT_BYTES} (inference-ready gate)")
else:
    print("[WARN] REQUIRE_INFERENCE_READY=0 — skipped readiness gate")

print("Smoke / check-input completed successfully.")
